# MNIST Image Recognition: MLP vs CNN vs Transformer

This Google Colab notebook trains a small neural network on the **official MNIST training split** and evaluates it on the **separate official MNIST test split**.

Choose one model in the configuration cell:
- `mlp`: shallow fully connected neural network
- `cnn`: convolutional neural network (recommended starting point)
- `transformer`: small Transformer encoder

It also compares clean test accuracy with accuracy on randomly rotated/translated test images. The augmented test result is a robustness check, not the official clean-test score.

## 1. Imports and configuration

In [21]:
import random
import torch
from torch import nn
from torch.utils.data import DataLoader
from torchvision import datasets, transforms

# Change these settings to experiment
MODEL_NAME = "transformer"       # "mlp", "cnn", or "transformer"
EPOCHS = 3
BATCH_SIZE = 128
LEARNING_RATE = 0.001
USE_TRAIN_AUGMENTATION = True
SEED = 42

random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
print("Selected model:", MODEL_NAME)

# In Colab, enable a GPU under Runtime > Change runtime type > T4 GPU (if available).

Device: cpu
Selected model: transformer


## 2. Define the models

In [22]:
class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.net = nn.Sequential(
            nn.Flatten(),
            nn.Linear(28 * 28, 256),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(256, 10),
        )

    def forward(self, x):
        return self.net(x)


class CNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.features = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 28x28 -> 14x14
            nn.Conv2d(32, 64, kernel_size=3, padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2),  # 14x14 -> 7x7
        )
        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Linear(64 * 7 * 7, 128),
            nn.ReLU(),
            nn.Dropout(0.25),
            nn.Linear(128, 10),
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class SmallVisionTransformer(nn.Module):
    def __init__(self):
        super().__init__()
        embed_dim = 64
        # Convert a 28x28 image into sixteen 7x7 patches.
        self.patch_embed = nn.Conv2d(1, embed_dim, kernel_size=7, stride=7)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, embed_dim))
        self.pos_embed = nn.Parameter(torch.zeros(1, 17, embed_dim))
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=4,
            dim_feedforward=128,
            dropout=0.1,
            batch_first=True,
            activation="gelu",
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=2)
        self.norm = nn.LayerNorm(embed_dim)
        self.head = nn.Linear(embed_dim, 10)

    def forward(self, x):
        x = self.patch_embed(x)            # [batch, 64, 4, 4]
        x = x.flatten(2).transpose(1, 2)   # [batch, 16, 64]
        cls = self.cls_token.expand(x.size(0), -1, -1)
        x = torch.cat((cls, x), dim=1)     # [batch, 17, 64]
        x = self.encoder(x + self.pos_embed)
        return self.head(self.norm(x[:, 0]))


MODEL_CLASSES = {
    "mlp": MLP,
    "cnn": CNN,
    "transformer": SmallVisionTransformer,
}
assert MODEL_NAME in MODEL_CLASSES, "MODEL_NAME must be mlp, cnn, or transformer"
model = MODEL_CLASSES[MODEL_NAME]().to(device)
print(model)

SmallVisionTransformer(
  (patch_embed): Conv2d(1, 64, kernel_size=(7, 7), stride=(7, 7))
  (encoder): TransformerEncoder(
    (layers): ModuleList(
      (0-1): 2 x TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=64, out_features=64, bias=True)
        )
        (linear1): Linear(in_features=64, out_features=128, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=128, out_features=64, bias=True)
        (norm1): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (norm2): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
  )
  (norm): LayerNorm((64,), eps=1e-05, elementwise_affine=True)
  (head): Linear(in_features=64, out_features=10, bias=True)
)


## 3. Download MNIST and prepare data

In [24]:
normalize = transforms.Normalize((0.1307,), (0.3081,))

if USE_TRAIN_AUGMENTATION:
    train_transform = transforms.Compose([
        transforms.RandomAffine(degrees=10, translate=(0.08, 0.08)),
        transforms.ToTensor(),
        normalize,
    ])
else:
    train_transform = transforms.Compose([
        transforms.ToTensor(),
        normalize,
    ])

test_transform = transforms.Compose([
    transforms.ToTensor(),
    normalize,
])

# Official training and test splits remain separate.
train_dataset = datasets.MNIST(
    root="/content/data", train=True, download=True, transform=train_transform
)
test_dataset = datasets.MNIST(
    root="/content/data", train=False, download=True, transform=test_transform
)

train_loader = DataLoader(
    train_dataset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2,
    pin_memory=torch.cuda.is_available()
)
test_loader = DataLoader(
    test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2,
    pin_memory=torch.cuda.is_available()
)

print("Training examples:", len(train_dataset))
print("Test examples:", len(test_dataset))

Training examples: 60000
Test examples: 10000


## 4. Training and evaluation functions

In [25]:
criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

def train_one_epoch(model, loader):
    model.train()
    total_loss = 0.0
    total = 0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        logits = model(images)
        loss = criterion(logits, labels)
        loss.backward()
        optimizer.step()
        total_loss += loss.item() * labels.size(0)
        total += labels.size(0)
    return total_loss / total


def evaluate(model, loader):
    model.eval()
    correct = 0
    total = 0
    total_loss = 0.0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            logits = model(images)
            loss = criterion(logits, labels)
            total_loss += loss.item() * labels.size(0)
            correct += (logits.argmax(dim=1) == labels).sum().item()
            total += labels.size(0)
    return 100.0 * correct / total, total_loss / total

## 5. Train the model and check the official test set

In [26]:
history = []

for epoch in range(1, EPOCHS + 1):
    train_loss = train_one_epoch(model, train_loader)
    test_accuracy, test_loss = evaluate(model, test_loader)
    history.append({
        "epoch": epoch,
        "train_loss": train_loss,
        "test_loss": test_loss,
        "test_accuracy_percent": test_accuracy,
    })
    print(
        f"Epoch {epoch}/{EPOCHS} | "
        f"train loss: {train_loss:.4f} | "
        f"test loss: {test_loss:.4f} | "
        f"clean test accuracy: {test_accuracy:.2f}%"
    )

Epoch 1/3 | train loss: 1.3898 | test loss: 0.5812 | clean test accuracy: 80.49%
Epoch 2/3 | train loss: 0.6558 | test loss: 0.2456 | clean test accuracy: 92.66%
Epoch 3/3 | train loss: 0.4558 | test loss: 0.1736 | clean test accuracy: 94.67%


## 6. Optional augmentation robustness experiment

In [27]:
# This applies random transformations to copies of the official test images.
# It is only an additional robustness check; do not use these images for training.
augmented_test_transform = transforms.Compose([
    transforms.RandomAffine(degrees=15, translate=(0.12, 0.12)),
    transforms.ToTensor(),
    normalize,
])
augmented_test_dataset = datasets.MNIST(
    root="/content/data", train=False, download=True,
    transform=augmented_test_transform
)
augmented_test_loader = DataLoader(
    augmented_test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=2
)

augmented_accuracy, augmented_loss = evaluate(model, augmented_test_loader)
clean_accuracy, clean_loss = evaluate(model, test_loader)

print(f"Official clean test accuracy: {clean_accuracy:.2f}%")
print(f"Augmented-image accuracy:     {augmented_accuracy:.2f}%")
print("Augmented accuracy is a robustness result, not the official clean-test score.")

Official clean test accuracy: 94.67%
Augmented-image accuracy:     82.98%
Augmented accuracy is a robustness result, not the official clean-test score.


## 7. Save model weights

In [8]:
model_path = f"/content/mnist_{MODEL_NAME}.pth"
torch.save(model.state_dict(), model_path)
print("Saved:", model_path)

# Optional: download the weights to your computer.
from google.colab import files
files.download(model_path)

Saved: /content/mnist_cnn.pth


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>